In [ ]:
import pandas as pd
import openpyxl
import glob
import pymupdf # PyMuPDF (formerly fitz)
import pdfplumber

In [ ]:
%pip install pdfplumber


   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ----------------- ---------------------- 2.9/6.6 MB 14.4 MB/s eta 0:00:01
   ---------------------------------- ----- 5.8/6.6 MB 13.9 MB/s eta 0:00:01
   ---------------------------------------- 6.6/6.6 MB 13.1 MB/s  0:00:00
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   --------------------- ------------------ 2.1/3.8 MB 9.8 MB/s eta 0:00:01
   ---------------------------------------- 3.8/3.8 MB 10.3 MB/s  0:00:00
   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   -------------- ------------------------- 2.6/7.2 MB 14.1 MB/s eta 0:00:01
   ------------------------------- -------- 5.8/7.2 MB 14.3 MB/s eta 0:00:01
   ---------------------------------------- 7.2/7.2 MB 13.0 MB/s  0:00:00
   ---------------------------------------- 0.0/3.9 MB ? eta -:--:--
   -------------------------- ------------- 2.6/3.9 MB 12.9 MB/s eta 0:00:01
   --------------------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [ ]:
ruta_carpeta = "Descargas/Q10/*.xlsx"
archivos_excel = glob.glob(ruta_carpeta)

In [ ]:
import pandas as pd
import glob

# 1. Obtener la lista de archivos
ruta_carpeta = "Descargas/Q10/*.xlsx"
archivos_excel = glob.glob(ruta_carpeta)

lista_dfs = []
palabras_glosario = ['GLOSARIO', 'I*', 'I', 'A']

# 2. Procesar cada archivo
for archivo in archivos_excel:
    df_meta = pd.read_excel(archivo, header=None, nrows=5)
    
    sede_jornada = df_meta.iloc[1, 3]
    periodo      = df_meta.iloc[2, 3]
    docente      = df_meta.iloc[3, 3]
    programa     = df_meta.iloc[1, 6]
    asignatura   = df_meta.iloc[2, 6]
    fecha_inicio = df_meta.iloc[3, 6]
    fecha_fin    = df_meta.iloc[3, 9]
    
    df = pd.read_excel(archivo, header=5)
    df = df.dropna(how='all', axis=1)
    df = df.drop(columns=['Unnamed: 0', 'Unnamed: 2', 'Unnamed: 3'], errors='ignore')
    
    df.columns = df.columns.str.strip().str.replace('Identificacion', 'Identificación')
    
    if 'Estudiantes' in df.columns:
        df = df[~df['Estudiantes'].astype(str).str.strip().isin(palabras_glosario)]
    
    if 'Identificación' in df.columns:
        df['Identificación'] = df['Identificación'].fillna('').astype(str).str.replace(r'[^\d]', '', regex=True).str.strip()
    
    cols_acum = ['ASIST.ACUM', 'INAS.ACUM']
    cols_existentes = [c for c in cols_acum if c in df.columns]
    df[cols_existentes] = df[cols_existentes].fillna(0).astype(int)
    
    df['Sede_Jornada'] = sede_jornada
    df['Periodo']      = periodo
    df['Docente']      = docente
    df['Programa']     = programa
    df['Asignatura']   = asignatura
    df['Fecha_Inicio'] = fecha_inicio
    df['Fecha_Fin']    = fecha_fin
    
    lista_dfs.append(df)

# 3. Concatenar
df_consolidado = pd.concat(lista_dfs, ignore_index=True)

# 4. ELIMINACIÓN DE NA / NAN / NULOS EN IDENTIFICACIÓN
df_consolidado['Identificación'] = df_consolidado['Identificación'].astype(str).str.strip()

df_consolidado = df_consolidado[
    df_consolidado['Identificación'].notna() & 
    ~df_consolidado['Identificación'].isin(['nan', 'NaN', 'None', ''])
]

# 5. Reordenar y Exportar
columnas_prioritarias = ['Identificación', 'Estudiantes', 'ASIST.ACUM', 'INAS.ACUM']
otras_columnas = [col for col in df_consolidado.columns if col not in columnas_prioritarias]
df_consolidado = df_consolidado[columnas_prioritarias + otras_columnas]

df_consolidado.to_excel("Consolidado_Inasistencias.xlsx", index=False)
print("¡Listo! Se eliminaron todas las filas con Identificación nula/NaN.")

¡Listo! Se eliminaron todas las filas con Identificación nula/NaN.


In [ ]:
df

,Estudiantes,Identificación,sept. 07 2026,sept. 08 2026,sept. 09 2026,sept. 10 2026,sept. 11 2026,sept. 14 2026,sept. 15 2026,sept. 16 2026,sept. 17 2026,ASIST.ACUM,INAS.ACUM,Sede_Jornada,Periodo,Docente,Programa,Asignatura,Fecha_Inicio,Fecha_Fin
0,Agudelo Rodas Gustavo Adolfo,80213830,A,A,A,A,A,A,A,A,A,9,0,B2G Única,B2G_YES BOGOTA-2 C1_SEP_2026,Ramírez Guiza Andrés Felipe,YES BOGOTA 2026-2,SDDE26-2_B1-1,7/09/2026,13/11/2026
1,Arboleda Arboleda Bladimir Bladimir,82395418,A,A,A,A,A,A,A,A,A,9,0,B2G Única,B2G_YES BOGOTA-2 C1_SEP_2026,Ramírez Guiza Andrés Felipe,YES BOGOTA 2026-2,SDDE26-2_B1-1,7/09/2026,13/11/2026
2,Ascencio Soba Juan Sebastian,1016954083,A,A,A,I,I,I,I,I,I,3,6,B2G Única,B2G_YES BOGOTA-2 C1_SEP_2026,Ramírez Guiza Andrés Felipe,YES BOGOTA 2026-2,SDDE26-2_B1-1,7/09/2026,13/11/2026
3,Avila Gutierrez Johan Santiago,1029281796,A,A,A,A,A,A,A,A,A,9,0,B2G Única,B2G_YES BOGOTA-2 C1_SEP_2026,Ramírez Guiza Andrés Felipe,YES BOGOTA 2026-2,SDDE26-2_B1-1,7/09/2026,13/11/2026
4,Barbosa Barbosa Ana Maria,1193075239,A,A,A,A,A,A,A,A,A,9,0,B2G Única,B2G_YES BOGOTA-2 C1_SEP_2026,Ramírez Guiza Andrés Felipe,YES BOGOTA 2026-2,SDDE26-2_B1-1,7/09/2026,13/11/2026
5,Bernate Hernandez Jhoan Camilo,1021392721,A,A,A,A,A,A,A,A,A,9,0,B2G Única,B2G_YES BOGOTA-2 C1_SEP_2026,Ramírez Guiza Andrés Felipe,YES BOGOTA 2026-2,SDDE26-2_B1-1,7/09/2026,13/11/2026
6,Bohorquez Montenegro Lida Lida,52933644,A,A,A,A,A,A,A,A,A,9,0,B2G Única,B2G_YES BOGOTA-2 C1_SEP_2026,Ramírez Guiza Andrés Felipe,YES BOGOTA 2026-2,SDDE26-2_B1-1,7/09/2026,13/11/2026
7,Castañeda Andrade Sandra Paola,1010760253,A,A,A,A,A,A,A,I,I,7,2,B2G Única,B2G_YES BOGOTA-2 C1_SEP_2026,Ramírez Guiza Andrés Felipe,YES BOGOTA 2026-2,SDDE26-2_B1-1,7/09/2026,13/11/2026
8,Chocontá Marín Wendy María,1030660512,A,A,A,A,I,I,A,A,A,7,2,B2G Única,B2G_YES BOGOTA-2 C1_SEP_2026,Ramírez Guiza Andrés Felipe,YES BOGOTA 2026-2,SDDE26-2_B1-1,7/09/2026,13/11/2026
9,Contreras Vallejo Vallejo Juan Nicolas Juan Ni...,1001274624,A,A,A,A,A,A,A,A,A,9,0,B2G Única,B2G_YES BOGOTA-2 C1_SEP_2026,Ramírez Guiza Andrés Felipe,YES BOGOTA 2026-2,SDDE26-2_B1-1,7/09/2026,13/11/2026


In [ ]:
df_detalle_matriculas = pd.read_excel("insumos\Detalle_actualizado_matriculas.xlsx")
df_detalle_matriculas.head()

<>:1: SyntaxWarning: invalid escape sequence '\D'
<>:1: SyntaxWarning: invalid escape sequence '\D'
C:\Users\sebastian.mendez\AppData\Local\Temp\ipykernel_26972\3694588752.py:1: SyntaxWarning: invalid escape sequence '\D'
  df_detalle_matriculas = pd.read_excel("insumos\Detalle_actualizado_matriculas.xlsx")


,Nombre estudiante,Tipo identificación,Número identificación,Teléfono,Celular,Correo electrónico,Condición matrícula,Código curso,Nombre curso,Asignatura,...,Código matrícula,Período,Fecha inicio curso,Fecha fin curso,Días horario,Hora inicio horario,Hora fin horario,Formalizado,Valor a pagar,Valor pagado
0,Aanchez Caicedo Maria Fer,Cédula de Ciudadanía,1091353058,3204852810,3204852810,mariafernandasanchez464@gmail.com,Nuevo,-5PM135 SDDE_2026-2,B1.2-5PM135 SDDE_2026-2,SDDE26-2_B1-2,...,149567,B2G_YES BOGOTA-2 C1_SEP_2026,2026-09-07,2026-11-13,"Lunes, Martes, Miércoles, Jueves, Viernes",5:20 p.m.,7:00 p.m.,Sí,709663,0
1,Abdor Pacheco Yessenia,Cédula de Ciudadanía,1014979993,3006974871,3006974871,61.yaaap@gmail.com,Nuevo,-6AM7 SDDE_2026-2,B1.1-6AM7 SDDE_2026-2,SDDE26-2_B1-1,...,146350,B2G_YES BOGOTA-2 C1_SEP_2026,2026-09-07,2026-11-13,"Lunes, Martes, Miércoles, Jueves, Viernes",6:10 a.m.,7:50 a.m.,Sí,709663,0
2,Abella Mora Edson David,Cédula de Ciudadanía,80791389,3196851651,3196851651,edsondavidmora@gmail.com,Nuevo,-8AM85 SDDE_2026-2,B1.2-8AM85 SDDE_2026-2,SDDE26-2_B1-2,...,150666,B2G_YES BOGOTA-2 C1_SEP_2026,2026-09-07,2026-11-13,"Lunes, Martes, Miércoles, Jueves, Viernes",8:10 a.m.,9:50 a.m.,Sí,709663,0
3,Abril Fonseca Manuel Fabian,Cédula de Ciudadanía,1015455378,3239428337,3239428337,mfabrilf@gmail.com,Nuevo,-7PM215 SDDE_2026-2,B2.1-7PM215 SDDE_2026-2,SDDE26-2_B2-1,...,89020,B2G_YES BOGOTA-2 C1_SEP_2026,2026-09-07,2026-11-13,"Lunes, Martes, Miércoles, Jueves, Viernes",7:10 p.m.,8:50 p.m.,Sí,709663,0
4,Abril Lancheros Maria Amparo,Cédula de Ciudadanía,35323031,3164467688,3164467688,solomusicarsa@gmail.com,Nuevo,-5PM108 SDDE_2026-2,B1.2-5PM108 SDDE_2026-2,SDDE26-2_B1-2,...,149218,B2G_YES BOGOTA-2 C1_SEP_2026,2026-09-07,2026-11-13,"Lunes, Martes, Miércoles, Jueves, Viernes",5:20 p.m.,7:00 p.m.,Sí,709663,0


In [ ]:
from pathlib import Path
import pdfplumber
import pandas as pd


def obtener_dataframe_estudiantes(ruta_pdf):
    """
    Lee un PDF con formato de tabla y devuelve un DataFrame de Pandas.
    
    :param ruta_pdf: str o Path con la ubicación del archivo PDF.
    :return: pd.DataFrame con la información procesada.
    """
    path = Path(ruta_pdf)
    
    # Validar que el archivo exista antes de procesar
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo PDF en la ruta: {path.resolve()}")

    todas_las_filas = []
    encabezados = None

    with pdfplumber.open(path) as pdf:
        for pagina in pdf.pages:
            tablas = pagina.extract_tables()
            
            for tabla in tablas:
                if not tabla:
                    continue
                
                # Definir los encabezados en la primera tabla encontrada
                if encabezados is None:
                    encabezados = [c.replace("\n", " ").strip() if c else "" for c in tabla[0]]
                    filas = tabla[1:]
                else:
                    filas = tabla

                # Limpiar saltos de línea e imprecisiones de texto en cada celda
                for fila in filas:
                    fila_limpia = [celda.replace("\n", " ").strip() if celda else "" for celda in fila]
                    
                    # Evitar duplicar encabezados si el reporte tiene múltiples páginas
                    if fila_limpia != encabezados:
                        todas_las_filas.append(fila_limpia)

    # Crear y retornar el DataFrame de Pandas
    return pd.DataFrame(todas_las_filas, columns=encabezados)

In [ ]:
df_matriculados = obtener_dataframe_estudiantes(r"insumos\Estudiantes Matriculados.pdf")

In [ ]:
df_matriculados.head()

,Estudiante,Identificación,Fecha Matrícula,Teléfono,Celular,Correo,Matrícula Formalizada,Nivel,Estado
0,Aanchez Caicedo Maria Fer,C.C. 1091353058,7/09/2026,3204852810,3204852810,mariafernandasanchez464@gmail.com,Sí,SDDE26_B 1-2,Nuevo
1,Abdor Pacheco Yessenia,C.C. 1014979993,5/09/2026,3006974871,3006974871,61.yaaap@gmail.com,Sí,SDDE26_B 1-1,Nuevo
2,Abella Martinez Vanessa,C.C. 1030690519,7/09/2026,3046576101,3046576101,vabemar18@gmail.com,Sí,SDDE26_B 1-2,Nuevo
3,Abella Mora Edson David,C.C. 80791389,8/09/2026,3196851651,3196851651,edsondavidmora@gmail.com,Sí,SDDE26_B 1-2,Nuevo
4,Abril Fonseca Manuel Fabian,C.C. 1015455378,8/09/2026,3239428337,3239428337,mfabrilf@gmail.com,Sí,SDDE26_B 2-1,Nuevo


In [ ]:
df_listado_matriculas = obtener_dataframe_estudiantes(r"insumos\ListadoMatriculas.pdf")

In [ ]:
df_listado_matriculas.head()

,Estudiante,Identificación,Fecha Renovación,Teléfono,Celular,Correo,Nivel,Estado
0,Segura Naranjo Danilo,C.C. 79718761,18/09/2026,3225990961,3225990961,dannynmd@hotmail.com,A1,Activo
1,Barrios Villa Luis Gustavo,C.C. 1082067550,18/09/2026,3166000615,3166000615,luis-gustavo30@hotmail.com,INTP_A1- 1_0610AM,Activo
2,Total Estudiantes Centro Única - INTENSIVO PRE...,,,,,,,2
3,Rodriguez Perea Yojari Liliana,C.C. 1018498749,18/09/2026,3023186535,3023186535,prliliana087@gmail.com,A1,Activo
4,Peña Ramirez Juan David,C.C. 1016113400,18/09/2026,3124304481,3124304481,jdpenax@hotmail.com,AWKP_A1- 2_0110PM,Activo


In [ ]:
df_detalle_matriculas.head()

,Nombre estudiante,Tipo identificación,Número identificación,Teléfono,Celular,Correo electrónico,Condición matrícula,Código curso,Nombre curso,Asignatura,...,Código matrícula,Período,Fecha inicio curso,Fecha fin curso,Días horario,Hora inicio horario,Hora fin horario,Formalizado,Valor a pagar,Valor pagado
0,Aanchez Caicedo Maria Fer,Cédula de Ciudadanía,1091353058,3204852810,3204852810,mariafernandasanchez464@gmail.com,Nuevo,-5PM135 SDDE_2026-2,B1.2-5PM135 SDDE_2026-2,SDDE26-2_B1-2,...,149567,B2G_YES BOGOTA-2 C1_SEP_2026,2026-09-07,2026-11-13,"Lunes, Martes, Miércoles, Jueves, Viernes",5:20 p.m.,7:00 p.m.,Sí,709663,0
1,Abdor Pacheco Yessenia,Cédula de Ciudadanía,1014979993,3006974871,3006974871,61.yaaap@gmail.com,Nuevo,-6AM7 SDDE_2026-2,B1.1-6AM7 SDDE_2026-2,SDDE26-2_B1-1,...,146350,B2G_YES BOGOTA-2 C1_SEP_2026,2026-09-07,2026-11-13,"Lunes, Martes, Miércoles, Jueves, Viernes",6:10 a.m.,7:50 a.m.,Sí,709663,0
2,Abella Mora Edson David,Cédula de Ciudadanía,80791389,3196851651,3196851651,edsondavidmora@gmail.com,Nuevo,-8AM85 SDDE_2026-2,B1.2-8AM85 SDDE_2026-2,SDDE26-2_B1-2,...,150666,B2G_YES BOGOTA-2 C1_SEP_2026,2026-09-07,2026-11-13,"Lunes, Martes, Miércoles, Jueves, Viernes",8:10 a.m.,9:50 a.m.,Sí,709663,0
3,Abril Fonseca Manuel Fabian,Cédula de Ciudadanía,1015455378,3239428337,3239428337,mfabrilf@gmail.com,Nuevo,-7PM215 SDDE_2026-2,B2.1-7PM215 SDDE_2026-2,SDDE26-2_B2-1,...,89020,B2G_YES BOGOTA-2 C1_SEP_2026,2026-09-07,2026-11-13,"Lunes, Martes, Miércoles, Jueves, Viernes",7:10 p.m.,8:50 p.m.,Sí,709663,0
4,Abril Lancheros Maria Amparo,Cédula de Ciudadanía,35323031,3164467688,3164467688,solomusicarsa@gmail.com,Nuevo,-5PM108 SDDE_2026-2,B1.2-5PM108 SDDE_2026-2,SDDE26-2_B1-2,...,149218,B2G_YES BOGOTA-2 C1_SEP_2026,2026-09-07,2026-11-13,"Lunes, Martes, Miércoles, Jueves, Viernes",5:20 p.m.,7:00 p.m.,Sí,709663,0


## Informe: Estudiantes notas y observaciones (Q10)

Una fila por estudiante, curso, corte y subparámetro (6.487 estudiantes x 10 = 64.870 filas).

La columna `Subparámetro` trae pegados el criterio y el corte (`Task accomplishment 1`), y además
está escrita distinto entre cortes (`accomplishment` en el 1 y `Accomplishment` en el 2). Se parte en:

- **Criterio**: Communication, Language, Learning, Task Accomplishment, Workshops
- **Corte**: 1 o 2

In [ ]:
df_notas = pd.read_excel(r"insumos\Estudiantes_notas_y_observaciones.xlsx")
df_notas.columns = df_notas.columns.str.strip()

# "Task accomplishment 1" -> criterio "Task accomplishment" + corte "1"
partes = (df_notas["Subparámetro"].astype(str).str.strip()
          .str.extract(r"^(?P<criterio>.+?)\s*(?P<corte>\d+)$"))

# .str.title() unifica la escritura: "accomplishment" y "Accomplishment" quedan igual
df_notas["Criterio"] = partes["criterio"].str.strip().str.title()
df_notas["Corte"] = pd.to_numeric(partes["corte"], errors="coerce").astype("Int64")

# El corte tambien viene en la columna Parámetro ("Corte 1"): se usa como respaldo y comprobacion
corte_parametro = pd.to_numeric(
    df_notas["Parámetro"].astype(str).str.extract(r"(\d+)")[0], errors="coerce").astype("Int64")
df_notas["Corte"] = df_notas["Corte"].fillna(corte_parametro)

# Dejar las columnas nuevas al lado del subparametro original
orden = list(df_notas.columns)
for columna in ("Criterio", "Corte"):
    orden.remove(columna)
posicion = orden.index("Subparámetro") + 1
df_notas = df_notas[orden[:posicion] + ["Criterio", "Corte"] + orden[posicion:]]

print(f"Filas: {len(df_notas)}")
print("Criterios:", sorted(df_notas["Criterio"].dropna().unique()))
print("Cortes:", sorted(df_notas["Corte"].dropna().unique()))
print("Filas que no se pudieron separar:", int(df_notas["Criterio"].isna().sum()))
print("Filas donde el corte no coincide con el parametro:", int((df_notas["Corte"] != corte_parametro).sum()))

In [ ]:
# Como quedo la separacion: las 10 combinaciones originales -> 5 criterios x 2 cortes
df_notas[["Parámetro", "Subparámetro", "Criterio", "Corte"]].drop_duplicates().sort_values(["Corte", "Criterio"])

In [ ]:
# 'Número identificación' llega como numero (int64), asi que filtrar con texto no encuentra nada.
# Se deja una columna 'Documento' de texto y sin puntos ni espacios, que ademas sirve para cruzar
# con los otros informes (en algunos viene como "C.C. 1091353058").
df_notas["Documento"] = (df_notas["Número identificación"].astype(str)
                         .str.replace(r"[^\d]", "", regex=True).str.strip())

# --- un estudiante ---
documento = "1091353058"
df_uno = df_notas[df_notas["Documento"] == documento]

# --- varios estudiantes ---
documentos = ["1091353058", "1000383076"]
df_varios = df_notas[df_notas["Documento"].isin(documentos)]

print(f"{documento}: {len(df_uno)} filas")
print(f"{len(documentos)} documentos: {len(df_varios)} filas de {df_varios['Documento'].nunique()} estudiantes")

df_uno[["Estudiante", "Documento", "Curso", "Docente", "Corte", "Criterio",
        "Nota subparámetro", "Inasistencias"]]

In [ ]:
# De 10 filas por estudiante a 2 (una por corte), con los 5 criterios como columnas.
# Se hace en dos pasos para no armar un pivote gigante: primero solo las notas, luego los demas datos.
claves = ["Documento", "Curso", "Corte"]

notas_por_criterio = (df_notas
                      .pivot(index=claves, columns="Criterio", values="Nota subparámetro")
                      .reset_index()
                      .rename_axis(columns=None))

# Estas columnas son iguales en las 5 filas del mismo estudiante-curso-corte, asi que basta una
datos_del_estudiante = (df_notas[claves + ["Estudiante", "Tipo identificación", "Sede - Jornada",
                                           "Programa", "Asignatura", "Período", "Código curso",
                                           "Docente", "Porcentaje parámetro", "Nota parámetro",
                                           "Promedio", "Acumulado", "Inasistencias", "Observaciones"]]
                        .drop_duplicates(subset=claves))

df_notas_ancho = datos_del_estudiante.merge(notas_por_criterio, on=claves, how="left")

print(f"{len(df_notas_ancho)} filas  =  {df_notas['Documento'].nunique()} estudiantes x 2 cortes")
df_notas_ancho[df_notas_ancho["Documento"] == "1000383076"][
    ["Documento", "Estudiante", "Curso", "Corte", "Communication", "Language", "Learning",
     "Task Accomplishment", "Workshops", "Nota parámetro", "Inasistencias"]]